# Assignment 2 — Question 3: Association Rule Mining (FP-growth)

Goal: mine frequent patterns and association rules on the subset of `mobile_price.csv` where `price_range == 1`, using only four features: `ram`, `int_memory`, `px_width`, `battery_power`.

Each feature is discretised into `low / medium / high` buckets using a **3 : 4 : 3** split of its value range (`max − min`).

In [2]:
# !pip install mlxtend
import os
import numpy as np
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

os.makedirs('chart', exist_ok=True)
pd.set_option('display.width', 160)
pd.set_option('display.max_colwidth', 80)

## Load data and filter to `price_range == 1`

In [3]:
df = pd.read_csv('data/mobile_price.csv')
print(f'Full dataset:    {df.shape}')

df_p1 = df[df['price_range'] == 1].copy()
print(f'price_range==1:  {df_p1.shape}')

FEATURES = ['ram', 'int_memory', 'px_width', 'battery_power']
df_p1[FEATURES].describe()

Full dataset:    (2000, 21)
price_range==1:  (500, 21)


,ram,int_memory,px_width,battery_power
count,500.000000,500.000000,500.000000,500.000000
mean,1679.490000,32.116000,1251.908000,1228.868000
std,465.850159,18.000739,433.564352,438.614528
min,387.000000,2.000000,500.000000,501.000000
25%,1354.000000,16.000000,878.750000,843.000000
50%,1686.500000,32.000000,1223.000000,1206.000000
75%,2033.750000,47.000000,1629.000000,1596.250000
max,2811.000000,64.000000,1998.000000,1996.000000


## Discretise each feature into low / medium / high (3 : 4 : 3 of the range)

For each feature `f` use `min(f)` and `max(f)` from the **filtered** subset:
- `low`     : `[min, min + 0.3*(max−min))`
- `medium`  : `[min + 0.3*(max−min), min + 0.7*(max−min))`
- `high`    : `[min + 0.7*(max−min), max]`

In [4]:
def discretise_3_4_3(series, name):
    lo, hi = series.min(), series.max()
    rng = hi - lo
    cut1 = lo + 0.3 * rng
    cut2 = lo + 0.7 * rng
    bins = [-np.inf, cut1, cut2, np.inf]
    labels = [f'{name}_low', f'{name}_medium', f'{name}_high']
    return pd.cut(series, bins=bins, labels=labels, right=False), (lo, cut1, cut2, hi)

discretised = {}
boundaries  = {}
for f in FEATURES:
    discretised[f], boundaries[f] = discretise_3_4_3(df_p1[f], f)

# Boundary table
bdf = pd.DataFrame(boundaries, index=['min', 'low|medium', 'medium|high', 'max']).T
print('=== Bin boundaries (per feature) ===')
print(bdf.round(2))

# Per-feature bucket counts
print('\n=== Bucket counts per feature ===')
for f in FEATURES:
    print(f'\n{f}:')
    print(discretised[f].value_counts().sort_index())

=== Bin boundaries (per feature) ===
                 min  low|medium  medium|high     max
ram            387.0      1114.2       2083.8  2811.0
int_memory       2.0        20.6         45.4    64.0
px_width       500.0       949.4       1548.6  1998.0
battery_power  501.0       949.5       1547.5  1996.0

=== Bucket counts per feature ===

ram:
ram
ram_low        53
ram_medium    341
ram_high      106
Name: count, dtype: int64

int_memory:
int_memory
int_memory_low       158
int_memory_medium    206
int_memory_high      136
Name: count, dtype: int64

px_width:
px_width
px_width_low       148
px_width_medium    208
px_width_high      144
Name: count, dtype: int64

battery_power:
battery_power
battery_power_low       154
battery_power_medium    207
battery_power_high      139
Name: count, dtype: int64


## Build transactions

Each row in the filtered dataset becomes one transaction containing four items, one per feature, e.g. `['ram_high', 'int_memory_low', 'px_width_low', 'battery_power_low']`.

In [5]:
transactions = []
for idx in df_p1.index:
    items = [str(discretised[f].loc[idx]) for f in FEATURES]
    transactions.append(items)

print(f'Number of transactions: {len(transactions)}')
print('First 5 transactions:')
for t in transactions[:5]:
    print('  ', t)

# One-hot encode for mlxtend
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
trans_df = pd.DataFrame(te_ary, columns=te.columns_)
trans_df.head()

Number of transactions: 500
First 5 transactions:
   ['ram_high', 'int_memory_low', 'px_width_low', 'battery_power_low']
   ['ram_medium', 'int_memory_medium', 'px_width_medium', 'battery_power_high']
   ['ram_low', 'int_memory_medium', 'px_width_high', 'battery_power_high']
   ['ram_medium', 'int_memory_medium', 'px_width_low', 'battery_power_high']
   ['ram_medium', 'int_memory_high', 'px_width_low', 'battery_power_medium']


,battery_power_high,battery_power_low,battery_power_medium,int_memory_high,int_memory_low,int_memory_medium,px_width_high,px_width_low,px_width_medium,ram_high,ram_low,ram_medium
0,False,True,False,False,True,False,False,True,False,True,False,False
1,True,False,False,False,False,True,False,False,True,False,False,True
2,True,False,False,False,False,True,True,False,False,False,True,False
3,True,False,False,False,False,True,False,True,False,False,False,True
4,False,False,True,True,False,False,False,True,False,False,False,True


## Q3(a) — Frequent patterns with `support ≥ 0.3` (FP-growth)

In [6]:
freq_patterns = fpgrowth(trans_df, min_support=0.3, use_colnames=True)
freq_patterns = freq_patterns.sort_values('support', ascending=False).reset_index(drop=True)
freq_patterns['itemset_size'] = freq_patterns['itemsets'].apply(len)
freq_patterns['itemsets_str'] = freq_patterns['itemsets'].apply(lambda s: ', '.join(sorted(s)))

print(f'Number of frequent patterns (support >= 0.3): {len(freq_patterns)}')
print()
print(freq_patterns[['itemset_size', 'support', 'itemsets_str']].to_string(index=False))

Number of frequent patterns (support >= 0.3): 8

 itemset_size  support                     itemsets_str
            1    0.682                       ram_medium
            1    0.416                  px_width_medium
            1    0.414             battery_power_medium
            1    0.412                int_memory_medium
            2    0.318 battery_power_medium, ram_medium
            1    0.316                   int_memory_low
            1    0.308                battery_power_low
            2    0.306      px_width_medium, ram_medium


## Q3(b) — Association rules with `support ≥ 0.3, confidence ≥ 0.4, lift ≥ 0.8`

In [7]:
# mlxtend's association_rules takes the frequent patterns and a single primary metric;
# we then filter by the remaining thresholds.
rules = association_rules(freq_patterns.drop(columns=['itemset_size', 'itemsets_str']),
                          metric='confidence', min_threshold=0.4)
rules = rules[(rules['support'] >= 0.3) & (rules['lift'] >= 0.8)].copy()
rules = rules.sort_values(['lift', 'confidence', 'support'], ascending=False).reset_index(drop=True)

rules['antecedents_str'] = rules['antecedents'].apply(lambda s: ', '.join(sorted(s)))
rules['consequents_str'] = rules['consequents'].apply(lambda s: ', '.join(sorted(s)))

print(f'Number of rules (support>=0.3, confidence>=0.4, lift>=0.8): {len(rules)}')
print()
print(rules[['antecedents_str', 'consequents_str', 'support', 'confidence', 'lift']]
      .round(4).to_string(index=False))

Number of rules (support>=0.3, confidence>=0.4, lift>=0.8): 4

     antecedents_str      consequents_str  support  confidence   lift
battery_power_medium           ram_medium    0.318      0.7681 1.1263
          ram_medium battery_power_medium    0.318      0.4663 1.1263
     px_width_medium           ram_medium    0.306      0.7356 1.0786
          ram_medium      px_width_medium    0.306      0.4487 1.0786


## Q3(c) — Observations

**Q3(a) summary:** 8 frequent patterns at `support ≥ 0.3` — **6 singletons + 2 pairs**. No 3- or 4-itemset clears the threshold.

**Q3(b) summary:** 4 rules survive `support ≥ 0.3, confidence ≥ 0.4, lift ≥ 0.8` — and all four involve `ram_medium`.

| antecedent | consequent | support | conf | lift |
|---|---|---:|---:|---:|
| battery_power_medium | ram_medium | 0.318 | 0.768 | 1.13 |
| ram_medium | battery_power_medium | 0.318 | 0.466 | 1.13 |
| px_width_medium | ram_medium | 0.306 | 0.736 | 1.08 |
| ram_medium | px_width_medium | 0.306 | 0.449 | 1.08 |

**Observations:**

1. **RAM is the defining spec of the mid-cheap segment.** `ram_medium` has support **0.682** — 68% of `price_range==1` phones fall in the middle 40% of the RAM range. None of the other features show that concentration; their three buckets sit close to the 30/40/30 split you'd expect from a uniform distribution. Manufacturers cluster RAM tightly around a "this is what a mid-cheap phone gets" value, while leaving battery, storage, and screen-width free to vary.

2. **The other three features are roughly uniform within the segment.** All non-RAM medium buckets are at 0.41–0.42 (the expected 0.40 from uniform), and `int_memory_low` (0.316) and `battery_power_low` (0.308) just barely clear the threshold — a mild low-skew but not strong.

3. **Every rule centers on `ram_medium`.** This is a *consequence* of RAM being so dominant, not an independent finding. A pair needs both items above 0.3 support; `ram_medium` is the only item frequent enough to participate in every qualifying pair.

4. **Confidence asymmetry is base-rate arithmetic.** `battery_power_medium → ram_medium` has confidence 0.77, but the reverse direction only 0.47. That's because `ram_medium` already sits at 68% baseline, so the reverse can't lift much above it. The symmetric measure here is **lift**, which is identical (1.13) in both directions.

5. **Lift values are close to 1 (1.08–1.13).** Real but weak associations: medium-battery and medium-screen do co-occur with medium-RAM slightly more than chance, but the coupling is mild. There is no tight engineering rule like "medium-spec phones always come with medium battery"; specs are largely independent within this price tier.

6. **Negative space.** No `_high` or `_low` itemset appears in any rule. The mid-cheap segment is internally homogeneous on RAM and otherwise diverse — nothing extreme co-occurs strongly enough to be mined at these thresholds.